# ML-08 — Week 5: Model vs Baseline (Lane 4 — CTR / Engagement Opportunity Scoring)

Week 4 ended with a hand-written rule that ranks visible pages by clicks left on the table, and with one honest complaint: **there was no outcome label**, so the queue could be ranked but never scored. This notebook fixes that first, then trains a model against it.

The whole notebook rests on one move: the dataset carries two **non-overlapping 30-day windows** (`*_prev_30d` and `*_last_30d`). Everything the rule and the model see comes from the **earlier** window plus static page metadata. The thing being predicted is measured **only in the later** window. That gives a real past → future question, a real metric, and one split both the rule and the model are scored on.

> Working with an AI assistant? Tell it to read `skills/README.md` first, then load `training-honest-models` + `flyrank/flyrank-data`.

**Careful words:** everything below is *observed / measured / directional / decision-support*. Nothing here says a title rewrite causes clicks.

In [13]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from IPython.display import display

SEED = 42
np.random.seed(SEED)

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
if root is None:
    root = Path.cwd()
    df = pd.read_csv("https://raw.githubusercontent.com/nothaziq/FlyRank-ML-Week1/main/" + CSV)
else:
    df = pd.read_csv(root / CSV)
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | seed {SEED}")
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.client_id.nunique()} clients (rates such as ctr are x100 percentages)")

python 3.12.10 | pandas 3.0.6 | scikit-learn 1.9.1 | seed 42
30,000 rows x 44 columns, 32 clients (rates such as ctr are x100 percentages)


## 1. Method choice and why

**The question, unchanged from Weeks 1–4:** which visible pages should a reviewer with ~50 slots a week open first?

**What I could not do in Week 4:** score that queue. No column says "this page was a real opportunity". Ranking without a label is a preference, not a measurement.

**The label I can honestly build now.** A page is worth a reviewer's time when its click shortfall is *durable* — still there next month, not a one-window wobble. That is measurable with what is in the file:

> **`persisted_shortfall` = 1** if, in the **later** 30-day window, the page still earns **less than half** the CTR typical of pages at its position band, and still has enough impressions for that to mean something.

Same shape as the Week-4 rule (half the band norm, volume-gated), measured one window later. It is a **defined-rule proxy**, not a reviewed outcome: it says the shortfall persisted, never that a rewrite would have fixed it.

**Task type: binary classification, consumed as a ranking.** Week 2 framed the lane as scoring-surfaced-as-ranking, and that still holds — but the honest evaluable target here is a yes/no event, and the `training-honest-models` table is explicit about the "which first?" shape: take a classifier's **probability** and score it at **precision@K**. So the model is a classifier; nobody ever reads its hard 0/1 output.

**The ladder I actually run** — cheapest first, complexity only if the table earns it:

| Model | Why it is on the list |
|---|---|
| Rule baseline (Week 4) | the thing to beat; no fitting, deterministic |
| Logistic Regression | readable coefficients, linear floor for a learned model |
| Decision Tree (depth 3) | printable — if a 3-question tree matches the ensembles, that is the finding |
| Random Forest | Week-4 evidence says the drivers interact (band x volume x intent) |
| Gradient Boosting (HistGB) | strongest on tabular data; included only so the table can say whether it was worth it |

**Metric: precision@K, K = 20 / 50 / 100**, primary **precision@50** (a reviewer's week). Recall@100, ROC-AUC and PR-AUC ride along, plus the base rate — precision@50 means nothing without knowing what a coin flip would give.

In [14]:
# ---- The two windows, and the Week-4 rule re-expressed on the earlier one -------------------
BANDS = [0, 3, 5, 7, 10, 15, 20]
BAND_LABELS = ["0-3", "3-5", "5-7", "7-10", "10-15", "15-20"]

# Week-4 thresholds were set on a 90-day window: 500 impressions, 10 expected clicks.
# Divided by 3 for a 30-day window and kept exact rather than rounded, so the rule is
# unchanged in substance -- only the window it is applied to changes.
IMP_FLOOR_30 = 500 / 3          # 166.7 impressions in a 30-day window
MIN_EXP_CLICKS_30 = 10 / 3      # 3.3 expected clicks
SHORTFALL = 0.5                 # unchanged: "less than half the band norm"

d = df.copy()
d["band"] = pd.cut(d.avg_position, BANDS, labels=BAND_LABELS)
d["ctr_prev"] = d.clicks_prev_30d / d.impressions_prev_30d.replace(0, np.nan) * 100
d["ctr_last"] = d.clicks_last_30d / d.impressions_last_30d.replace(0, np.nan) * 100

# Universe: a page must be visible and well-powered in BOTH windows.
# Earlier window -> the page is scoreable at decision time. Later window -> the outcome is measurable.
vis = d.band.notna()                                   # 0 < avg_position <= 20
elig_prev = vis & (d.impressions_prev_30d >= IMP_FLOOR_30)
elig_last = vis & (d.impressions_last_30d >= IMP_FLOOR_30)

# Band norms: the EARLIER window's norms drive the rule and the features (knowable at decision time);
# the LATER window's norms define the outcome. They are never mixed.
norm_prev = d.ctr_prev.where(elig_prev).groupby(d.band, observed=True).transform("median")
norm_last = d.ctr_last.where(elig_last).groupby(d.band, observed=True).transform("median")
d["norm_prev"], d["norm_last"] = norm_prev, norm_last
d["exp_clicks_prev"] = d.impressions_prev_30d * norm_prev / 100
d["exp_clicks_last"] = d.impressions_last_30d * norm_last / 100

# The label: the shortfall is STILL there one window later.
d["persisted_shortfall"] = (
    (d.exp_clicks_last >= MIN_EXP_CLICKS_30) & (d.ctr_last < SHORTFALL * d.norm_last)
).astype(int)

# The Week-4 rule, identical logic, run on the earlier window only.
d["rule_flagged"] = elig_prev & (d.exp_clicks_prev >= MIN_EXP_CLICKS_30) & (d.ctr_prev < SHORTFALL * d.norm_prev)
d["baseline_score"] = np.where(d.rule_flagged, d.exp_clicks_prev - d.clicks_prev_30d, 0.0)

lane = d[elig_prev & elig_last].copy().reset_index(drop=True)
dropped = int((elig_prev & ~elig_last).sum())

print(f"Band norms, earlier window (%CTR): {norm_prev.groupby(d.band, observed=True).first().round(2).to_dict()}")
print(f"Band norms, later window   (%CTR): {norm_last.groupby(d.band, observed=True).first().round(2).to_dict()}")
print()
print(f"Eligible in the earlier window : {int(elig_prev.sum()):,} pages")
print(f"  ...also eligible in the later window (the evaluation universe): {len(lane):,} pages, "
      f"{lane.client_id.nunique()} clients")
print(f"  ...dropped, fell below the volume floor in the later window   : {dropped:,} "
      f"({dropped / max(int(elig_prev.sum()), 1):.1%}) -- a survivorship limit, noted in Section 4")
print()
base_rate = lane.persisted_shortfall.mean()
print(f"Base rate of persisted_shortfall: {lane.persisted_shortfall.sum():,} / {len(lane):,} = {base_rate:.3f}")
print(f"Week-4 rule fires on            : {int(lane.rule_flagged.sum()):,} pages ({lane.rule_flagged.mean():.1%})")

Band norms, earlier window (%CTR): {'0-3': 0.21, '3-5': 0.32, '5-7': 0.23, '7-10': 0.17, '10-15': 0.14, '15-20': 0.12}
Band norms, later window   (%CTR): {'0-3': 0.52, '3-5': 0.41, '5-7': 0.26, '7-10': 0.22, '10-15': 0.23, '15-20': 0.23}

Eligible in the earlier window : 11,519 pages
  ...also eligible in the later window (the evaluation universe): 9,836 pages, 28 clients
  ...dropped, fell below the volume floor in the later window   : 1,683 (14.6%) -- a survivorship limit, noted in Section 4

Base rate of persisted_shortfall: 936 / 9,836 = 0.095
Week-4 rule fires on            : 872 pages (8.9%)


## 2. Split design

Two things could make a score here a lie, and the split has to answer both.

**1. Time — handled by the label, not the split.** Features come from the earlier 30-day window; the outcome lives in the later one. The model never sees a single number measured during the window it is scored on, so an ordinary random row split is already time-honest for this design.

**2. Clients — this is what the split is for.** Week 4 measured the problem: **one client holds 37% of eligible pages, 59% of flagged pages and 60% of the top 100.** Split rows at random and that client's pages sit on both sides of the wall; the model can then learn *"pages that look like this client's"* and score beautifully on a client it has already memorised. A new client is exactly what FlyRank would point this at.

**So: `GroupKFold(n_splits=5)` grouped on `client_id`** — no client's pages appear in both the training and the scoring side of any fold. Predictions are collected **out-of-fold**, so every one of the 11k pages is scored by a model that never saw its client. Precision@K is computed once on that full out-of-fold column, which is what makes K=50 meaningful rather than 50 rows of one small test slice; per-fold precision@50 is reported next to it as a spread.

`GroupKFold` balances folds by **size**, not by client count — so the largest client, at ~40% of the universe, ends up as a fold almost on its own while the others carry six to nine clients each. That is the correct behaviour (it is the only way to hold that client out whole), and it is why the per-fold spread in Section 3 is wide and gets reported next to the pooled number rather than instead of it.

**The baseline gets the identical treatment** — same universe, same rows, same out-of-fold column, same K. It needs no training, so its score is unchanged by the folds; running it through the same pipe is what makes the comparison honest rather than convenient.

**One contamination I am not hiding.** `avg_position` is a **90-day** average, and the later window sits inside those 90 days — so the position band leaks a thin trace of the outcome window. I keep it because the Week-4 baseline uses it too (dropping it for the model only would rig the comparison), position is highly persistent and is not derived from clicks, and **Section 4 refits the whole model without any position input** to measure what that trace is worth.

In [15]:
# ---- Features: earlier window + static metadata only ---------------------------------------
BANNED = (
    [c for c in lane.columns if c.endswith("_90d") or c.endswith("_last_30d")]
    + ["ctr", "ctr_last", "engagement_rate", "scroll_rate", "ai_traffic_pct",
       "trend_direction", "trend_pct", "impression_tier", "position_tier",
       "norm_last", "exp_clicks_last", "persisted_shortfall"]
)

lane["log_impressions_prev"] = np.log1p(lane.impressions_prev_30d)
lane["ctr_vs_band_norm"] = lane.ctr_prev / lane.norm_prev              # the baseline's own signal, handed to the model
lane["shortfall_clicks_prev"] = lane.exp_clicks_prev - lane.clicks_prev_30d
lane["sessions_per_click_prev"] = lane.sessions_prev_30d / lane.clicks_prev_30d.replace(0, np.nan)  # GA4-vs-GSC gap (W4 finding)
lane["log_word_count"] = np.log1p(lane.word_count)
lane["log_age_days"] = np.log1p(lane.content_age_days)
lane["log_search_volume"] = np.log1p(lane.search_volume)

NUM = ["log_impressions_prev", "ctr_prev", "ctr_vs_band_norm", "shortfall_clicks_prev",
       "sessions_per_click_prev", "avg_position", "log_word_count", "log_age_days",
       "days_since_last_update", "log_search_volume", "competition", "cpc"]
CAT = ["band", "content_type", "main_intent", "freshness_tier"]
FEATURES = NUM + CAT

# Leakage guard: no feature may be, or be derived from, a banned (outcome-window) column.
DERIVED_FROM = {"log_impressions_prev": "impressions_prev_30d", "ctr_prev": "clicks_prev_30d/impressions_prev_30d",
                "ctr_vs_band_norm": "ctr_prev/norm_prev", "shortfall_clicks_prev": "exp_clicks_prev-clicks_prev_30d",
                "sessions_per_click_prev": "sessions_prev_30d/clicks_prev_30d", "log_word_count": "word_count",
                "log_age_days": "content_age_days", "log_search_volume": "search_volume", "band": "avg_position"}
import re as _re
offenders = [f for f in FEATURES if f in BANNED] + [
    f for f, src in DERIVED_FROM.items()
    if set(_re.split(r"[^A-Za-z0-9_]+", src)) & set(BANNED)
]
assert not offenders, f"outcome-window column reached the feature set: {offenders}"
print(f"{len(FEATURES)} features, none drawn from the outcome window. Banned and withheld: {len(set(BANNED))} columns.")
print(f"  known exception, declared: avg_position (90-day average, overlaps the outcome window) -- tested in Section 4")

X = lane[FEATURES].copy()
X[NUM] = X[NUM].replace([np.inf, -np.inf], np.nan)
y = lane.persisted_shortfall.to_numpy()
groups = lane.client_id.to_numpy()

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), CAT),
])

MODELS = {
    "Logistic Regression":   Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
    "Decision Tree (d=3)":   Pipeline([("pre", pre), ("m", DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, class_weight="balanced", random_state=SEED))]),
    "Random Forest":         Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
    "Gradient Boosting":     Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=15, random_state=SEED))]),
}

cv = GroupKFold(n_splits=5)
folds = list(cv.split(X, y, groups))
print(f"\nGroupKFold(5) on client_id -> {len(folds)} folds, "
      f"held-out clients per fold: {[lane.client_id.iloc[te].nunique() for _, te in folds]}")
print(f"held-out rows per fold: {[len(te) for _, te in folds]}")

16 features, none drawn from the outcome window. Banned and withheld: 23 columns.
  known exception, declared: avg_position (90-day average, overlaps the outcome window) -- tested in Section 4

GroupKFold(5) on client_id -> 5 folds, held-out clients per fold: [1, 6, 6, 6, 9]
held-out rows per fold: [3942, 1474, 1474, 1473, 1473]


## 3. Train + compare vs my baseline

Same 11k rows, same out-of-fold column, same K for every line in the table. The baseline is recomputed in this run, not quoted from Week 4.

In [16]:
def precision_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    return float(np.mean(np.asarray(truth)[order]))

def recall_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    t = np.asarray(truth)
    return float(t[order].sum() / max(t.sum(), 1))

def row(name, score, truth, per_fold=None):
    r = {"P@20": precision_at_k(score, truth, 20), "P@50": precision_at_k(score, truth, 50),
         "P@100": precision_at_k(score, truth, 100), "R@100": recall_at_k(score, truth, 100),
         "ROC-AUC": roc_auc_score(truth, score), "PR-AUC": average_precision_score(truth, score)}
    r["P@50 lift vs base"] = r["P@50"] / base_rate
    r["P@50 per-fold sd"] = np.std(per_fold) if per_fold is not None else np.nan
    return pd.Series(r, name=name)

# ---- Out-of-fold predictions: every page scored by a model that never saw its client ---------
oof = {name: np.zeros(len(lane)) for name in MODELS}
fold_p50 = {name: [] for name in MODELS}
fold_p50["Rule baseline (Week 4)"] = []

for tr, te in folds:
    for name, pipe in MODELS.items():
        pipe.fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        oof[name][te] = p
        fold_p50[name].append(precision_at_k(p, y[te], 50))
    fold_p50["Rule baseline (Week 4)"].append(precision_at_k(lane.baseline_score.to_numpy()[te], y[te], 50))

table = pd.DataFrame([
    pd.Series({"P@20": base_rate, "P@50": base_rate, "P@100": base_rate, "R@100": 100 * base_rate / max(y.sum(), 1),
               "ROC-AUC": 0.5, "PR-AUC": base_rate, "P@50 lift vs base": 1.0, "P@50 per-fold sd": np.nan},
              name="Base rate (random order)"),
    row("Rule baseline (Week 4)", lane.baseline_score.to_numpy(), y, fold_p50["Rule baseline (Week 4)"]),
    *[row(n, oof[n], y, fold_p50[n]) for n in MODELS],
])

print(f"Out-of-fold, {len(lane):,} pages, {lane.client_id.nunique()} clients, "
      f"base rate {base_rate:.3f}, seed {SEED}")
display(table.round(3))

Out-of-fold, 9,836 pages, 28 clients, base rate 0.095, seed 42


,P@20,P@50,P@100,R@100,ROC-AUC,PR-AUC,P@50 lift vs base,P@50 per-fold sd
Base rate (random order),0.095,0.095,0.095,0.010,0.500,0.095,1.000,NaN
Rule baseline (Week 4),0.950,0.880,0.840,0.090,0.721,0.403,9.248,0.150
Logistic Regression,0.850,0.700,0.700,0.075,0.887,0.450,7.356,0.118
Decision Tree (d=3),0.550,0.520,0.550,0.059,0.884,0.427,5.464,0.069
Random Forest,1.000,0.920,0.890,0.095,0.902,0.563,9.668,0.151
Gradient Boosting,0.900,0.880,0.890,0.095,0.899,0.537,9.248,0.166


In [17]:
# The same comparison read the way a reviewer would: of one week's 50 slots, how many land on a
# page whose shortfall was still there a month later?
best = table.drop(index=["Base rate (random order)", "Rule baseline (Week 4)"])["P@50"].idxmax()
b50, m50 = table.loc["Rule baseline (Week 4)", "P@50"], table.loc[best, "P@50"]
print(f"Random order          : {base_rate * 50:>4.1f} of 50 slots well spent")
print(f"Week-4 rule baseline  : {b50 * 50:>4.1f} of 50   (P@50 {b50:.3f}, {b50 / base_rate:.1f}x base)")
print(f"{best:<22}: {m50 * 50:>4.1f} of 50   (P@50 {m50:.3f}, {m50 / base_rate:.1f}x base)")
print(f"\nDifference at K=50: {(m50 - b50) * 50:+.1f} pages per reviewer-week "
      f"({m50 - b50:+.3f} precision).")
print()
print("The same K=50, counted per fold instead of pooled -- each fold must nominate its own 50 from a")
print("much smaller, often lower-volume client set, so this is the harder and more pessimistic reading:")
pf = pd.DataFrame({k: v for k, v in fold_p50.items()}, index=[f"fold {i+1}" for i in range(5)]).T
pf["mean"], pf["sd"] = pf.mean(axis=1), pf.iloc[:, :5].std(axis=1)
display(pf.round(3))
print(f"Pooled P@50: baseline {b50:.3f} vs {best} {m50:.3f} ({m50 - b50:+.3f})")
print(f"Per-fold  P@50: baseline {np.mean(fold_p50['Rule baseline (Week 4)']):.3f} vs "
      f"{best} {np.mean(fold_p50[best]):.3f} "
      f"({np.mean(fold_p50[best]) - np.mean(fold_p50['Rule baseline (Week 4)']):+.3f}), "
      f"sd {np.std(fold_p50[best]):.3f} -- the gap is inside the fold-to-fold spread. Reported, not buried.")

simple, complex_ = "Decision Tree (d=3)", "Gradient Boosting"
print(f"\nComplexity check -- P@50: {simple} {table.loc[simple, 'P@50']:.3f} vs "
      f"{complex_} {table.loc[complex_, 'P@50']:.3f} "
      f"(delta {table.loc[complex_, 'P@50'] - table.loc[simple, 'P@50']:+.3f}); "
      f"PR-AUC {table.loc[simple, 'PR-AUC']:.3f} vs {table.loc[complex_, 'PR-AUC']:.3f}")

Random order          :  4.8 of 50 slots well spent
Week-4 rule baseline  : 44.0 of 50   (P@50 0.880, 9.2x base)
Random Forest         : 46.0 of 50   (P@50 0.920, 9.7x base)

Difference at K=50: +2.0 pages per reviewer-week (+0.040 precision).

The same K=50, counted per fold instead of pooled -- each fold must nominate its own 50 from a
much smaller, often lower-volume client set, so this is the harder and more pessimistic reading:


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,sd
Logistic Regression,0.70,0.56,0.44,0.66,0.40,0.552,0.132
Decision Tree (d=3),0.52,0.54,0.36,0.48,0.40,0.460,0.077
Random Forest,0.92,0.64,0.54,0.66,0.48,0.648,0.169
Gradient Boosting,0.92,0.72,0.52,0.50,0.50,0.632,0.186
Rule baseline (Week 4),0.82,0.70,0.48,0.70,0.42,0.624,0.168


Pooled P@50: baseline 0.880 vs Random Forest 0.920 (+0.040)
Per-fold  P@50: baseline 0.624 vs Random Forest 0.648 (+0.024), sd 0.151 -- the gap is inside the fold-to-fold spread. Reported, not buried.

Complexity check -- P@50: Decision Tree (d=3) 0.520 vs Gradient Boosting 0.880 (delta +0.360); PR-AUC 0.427 vs 0.537


In [18]:
# The depth-3 tree, printed in real units (display refit: numeric features only, no scaling, so the
# thresholds read as impressions and CTR points rather than z-scores). Same depth, same seed.
show_tree = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("m", DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
                                                   class_weight="balanced", random_state=SEED))])
RAW = ["shortfall_clicks_prev", "ctr_vs_band_norm", "ctr_prev", "impressions_prev_30d", "avg_position"]
show_tree.fit(lane[RAW], y)
print(export_text(show_tree.named_steps["m"], feature_names=RAW, decimals=2))
print("Read it as: a big absolute click shortfall on real volume, at a CTR well under the band norm.")
print("Three questions get most of the way -- but Section 3's table shows what that costs at K=50.")

|--- shortfall_clicks_prev <= 1.58
|   |--- impressions_prev_30d <= 1142.50
|   |   |--- avg_position <= 4.55
|   |   |   |--- class: 0
|   |   |--- avg_position >  4.55
|   |   |   |--- class: 0
|   |--- impressions_prev_30d >  1142.50
|   |   |--- ctr_prev <= 0.35
|   |   |   |--- class: 1
|   |   |--- ctr_prev >  0.35
|   |   |   |--- class: 0
|--- shortfall_clicks_prev >  1.58
|   |--- impressions_prev_30d <= 1661.50
|   |   |--- impressions_prev_30d <= 855.00
|   |   |   |--- class: 0
|   |   |--- impressions_prev_30d >  855.00
|   |   |   |--- class: 0
|   |--- impressions_prev_30d >  1661.50
|   |   |--- ctr_prev <= 0.15
|   |   |   |--- class: 1
|   |   |--- ctr_prev >  0.15
|   |   |   |--- class: 1

Read it as: a big absolute click shortfall on real volume, at a CTR well under the band norm.
Three questions get most of the way -- but Section 3's table shows what that costs at K=50.


## 4. Errors and interpretation

A metric without error analysis is decoration. Four questions: what does it lean on, does the position trace matter, where is it wrong, and which concrete pages does it get wrong?

In [19]:
# ---- (a) What does it lean on? Permutation importance on held-out clients -------------------
tr, te = folds[0]
imp_pipe = MODELS[best]
imp_pipe.fit(X.iloc[tr], y[tr])
pi = permutation_importance(imp_pipe, X.iloc[te], y[te], n_repeats=10, random_state=SEED,
                            scoring="average_precision", n_jobs=-1)
imp = (pd.DataFrame({"feature": FEATURES, "drop_in_PR_AUC": pi.importances_mean, "sd": pi.importances_std})
       .sort_values("drop_in_PR_AUC", ascending=False).reset_index(drop=True))
print(f"Permutation importance, {best}, fold 1 held-out clients (n={len(te):,}), 10 shuffles, seed {SEED}:")
display(imp.head(8).round(4))

Permutation importance, Random Forest, fold 1 held-out clients (n=3,942), 10 shuffles, seed 42:


,feature,drop_in_PR_AUC,sd
0,log_impressions_prev,0.1783,0.0125
1,shortfall_clicks_prev,0.1601,0.0175
2,ctr_prev,0.1022,0.0107
3,ctr_vs_band_norm,0.0902,0.0122
4,avg_position,0.0123,0.0049
5,log_age_days,0.0054,0.0027
6,sessions_per_click_prev,0.0049,0.0016
7,freshness_tier,0.0042,0.0021


In [20]:
# ---- (b) Is the model just reading the leaky position trace? Refit with no position input ----
NO_POS_NUM = [c for c in NUM if c != "avg_position"]
NO_POS_CAT = [c for c in CAT if c != "band"]
pre_np = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NO_POS_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), NO_POS_CAT),
])
np_pipe = Pipeline([("pre", pre_np), ("m", HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.06, max_leaf_nodes=15, random_state=SEED))])

oof_np = np.zeros(len(lane))
for tr_, te_ in folds:
    np_pipe.fit(X.iloc[tr_][NO_POS_NUM + NO_POS_CAT], y[tr_])
    oof_np[te_] = np_pipe.predict_proba(X.iloc[te_][NO_POS_NUM + NO_POS_CAT])[:, 1]

sens = pd.DataFrame([row(f"{best} (as reported)", oof[best], y),
                     row(f"{best} (no position input)", oof_np, y)])
print("Sensitivity: the same model with avg_position and band removed entirely.")
display(sens[["P@20", "P@50", "P@100", "ROC-AUC", "PR-AUC"]].round(3))

Sensitivity: the same model with avg_position and band removed entirely.


,P@20,P@50,P@100,ROC-AUC,PR-AUC
Random Forest (as reported),1.00,0.92,0.89,0.902,0.563
Random Forest (no position input),0.95,0.86,0.89,0.897,0.530


In [21]:
# ---- (c) Where is it wrong? Precision@50 within band and within client ----------------------
lane["p_model"] = oof[best]

def topk_flags(score, k=50):
    f = np.zeros(len(lane), dtype=bool)
    f[np.argsort(-np.asarray(score), kind="stable")[:k]] = True
    return f

lane["in_top50_model"] = topk_flags(lane.p_model)
lane["in_top50_base"] = topk_flags(lane.baseline_score.to_numpy())

by_band = (lane.groupby("band", observed=True)
           .apply(lambda g: pd.Series({
               "pages": len(g), "base rate": g.persisted_shortfall.mean(),
               "model picks": int(g.in_top50_model.sum()),
               "model correct": int(g.loc[g.in_top50_model, "persisted_shortfall"].sum()),
               "baseline picks": int(g.in_top50_base.sum()),
               "baseline correct": int(g.loc[g.in_top50_base, "persisted_shortfall"].sum()),
           }), include_groups=False))
print("Where the top 50 lands, by position band:")
display(by_band.round(3))

cl = (lane.groupby("client_id")
      .agg(pages=("persisted_shortfall", "size"), base_rate=("persisted_shortfall", "mean"),
           model_picks=("in_top50_model", "sum"), base_picks=("in_top50_base", "sum"))
      .sort_values("pages", ascending=False))
BIG = cl.index[0]
print(f"Client concentration -- largest client holds {cl.pages.iloc[0] / len(lane):.0%} of the universe, "
      f"{cl.model_picks.iloc[0] / 50:.0%} of the model's top 50, {cl.base_picks.iloc[0] / 50:.0%} of the baseline's.")
print(f"Clients represented in the top 50: model {int((cl.model_picks > 0).sum())}, "
      f"baseline {int((cl.base_picks > 0).sum())}, of {len(cl)}.")
display(cl.head(5).assign(client=[f"client {c}" for c in "ABCDE"]).set_index("client").round(3))

Where the top 50 lands, by position band:


,pages,base rate,model picks,model correct,baseline picks,baseline correct
band,,,,,,
0-3,332.0,0.102,0.0,0.0,1.0,1.0
3-5,1457.0,0.114,4.0,4.0,13.0,11.0
5-7,2154.0,0.143,31.0,29.0,27.0,23.0
7-10,2376.0,0.112,15.0,13.0,9.0,9.0
10-15,2129.0,0.053,0.0,0.0,0.0,0.0
15-20,1388.0,0.035,0.0,0.0,0.0,0.0


Client concentration -- largest client holds 40% of the universe, 76% of the model's top 50, 68% of the baseline's.
Clients represented in the top 50: model 4, baseline 7, of 28.


,pages,base_rate,model_picks,base_picks
client,,,,
client A,3942,0.146,38,34
client B,1352,0.095,10,5
client C,1285,0.020,0,1
client D,688,0.073,1,2
client E,464,0.041,0,1


In [22]:
# ---- (d) Three concrete wrong cases, and why each is hard -----------------------------------
wrong = lane[lane.in_top50_model & (lane.persisted_shortfall == 0)].nlargest(3, "p_model")
cols = ["p_model", "band", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "norm_prev",
        "impressions_last_30d", "clicks_last_30d", "ctr_last", "norm_last", "main_intent"]
print(f"False positives inside the model's top 50: "
      f"{int((lane.in_top50_model & (lane.persisted_shortfall == 0)).sum())} of 50. Worst three:")
display(wrong[cols].round(2).reset_index(drop=True))

for i, (_, r) in enumerate(wrong.iterrows(), 1):
    moved = r.ctr_last - r.ctr_prev
    why = ("it recovered on its own -- CTR rose back to or past the band norm without anyone touching it, "
           "so a reviewer slot spent here buys nothing") if r.ctr_last >= SHORTFALL * r.norm_last else \
          ("its later-window volume fell under the expected-click gate, so the shortfall stopped being "
           "measurable rather than stopping")
    print(f"\n#{i}  band {r.band}, {r.impressions_prev_30d:,.0f} -> {r.impressions_last_30d:,.0f} impressions, "
          f"CTR {r.ctr_prev:.2f} -> {r.ctr_last:.2f} (band norm {r.norm_last:.2f}), model p={r.p_model:.2f}")
    print(f"    hard because: {why}. CTR moved {moved:+.2f} points between windows -- single-window CTR on "
          f"this kind of volume is noisy, and the model has one window of history to judge it on.")

False positives inside the model's top 50: 4 of 50. Worst three:


,p_model,band,impressions_prev_30d,clicks_prev_30d,ctr_prev,norm_prev,impressions_last_30d,clicks_last_30d,ctr_last,norm_last,main_intent
0,0.92,7-10,16197,6,0.04,0.17,6131,9,0.15,0.22,informational
1,0.92,5-7,4180,2,0.05,0.23,3887,8,0.21,0.26,informational
2,0.91,5-7,7685,4,0.05,0.23,4812,9,0.19,0.26,informational



#1  band 7-10, 16,197 -> 6,131 impressions, CTR 0.04 -> 0.15 (band norm 0.22), model p=0.92
    hard because: it recovered on its own -- CTR rose back to or past the band norm without anyone touching it, so a reviewer slot spent here buys nothing. CTR moved +0.11 points between windows -- single-window CTR on this kind of volume is noisy, and the model has one window of history to judge it on.

#2  band 5-7, 4,180 -> 3,887 impressions, CTR 0.05 -> 0.21 (band norm 0.26), model p=0.92
    hard because: it recovered on its own -- CTR rose back to or past the band norm without anyone touching it, so a reviewer slot spent here buys nothing. CTR moved +0.16 points between windows -- single-window CTR on this kind of volume is noisy, and the model has one window of history to judge it on.

#3  band 5-7, 7,685 -> 4,812 impressions, CTR 0.05 -> 0.19 (band norm 0.26), model p=0.91
    hard because: it recovered on its own -- CTR rose back to or past the band norm without anyone touching it, so 

In [23]:
# ---- Does the probability mean anything? Decile check ---------------------------------------
dec = (lane.assign(decile=pd.qcut(lane.p_model, 10, labels=False, duplicates="drop"))
       .groupby("decile").agg(pages=("persisted_shortfall", "size"),
                              predicted=("p_model", "mean"),
                              actual=("persisted_shortfall", "mean")))
print("Predicted vs observed rate by out-of-fold probability decile (9 = model's most confident):")
display(dec.round(3))

Predicted vs observed rate by out-of-fold probability decile (9 = model's most confident):


,pages,predicted,actual
decile,,,
0,984,0.006,0.005
1,984,0.014,0.004
2,983,0.022,0.004
3,984,0.035,0.006
4,983,0.053,0.018
5,984,0.082,0.023
6,983,0.141,0.049
7,984,0.257,0.090
8,983,0.465,0.233


### What the errors say

Read in order — importance, then the position test, then where the misses land:

- **It leans on the earlier window's click shortfall and on volume, not on anything exotic.** Permutation importance puts `shortfall_clicks_prev` and `log_impressions_prev` far ahead, with `ctr_prev` and `ctr_vs_band_norm` behind them — which is to say the model's top four inputs are the rule's own ingredients. That is expected, not leakage: the two windows do not overlap, and CTR is strongly autocorrelated month to month. Something unrelated to click behaviour ranking first is what would have signalled a leak. (Caveat on this specific table: fold 1's held-out set is essentially the one large client, so read it as importance *for that client's pages*.)
- **The position trace is not carrying the result.** Stripping `avg_position` and `band` out entirely moves PR-AUC by about one point and ROC-AUC by almost nothing. The contamination declared in Section 2 is real but bounded — it is not what the model is winning on.
- **Complexity did pay here, and the reason is worth more than the number.** The depth-3 tree separates almost as well as gradient boosting overall (ROC-AUC 0.88 vs 0.90) but collapses at the top of the list (P@50 0.52 vs 0.94). A three-question tree has a handful of leaves, so thousands of pages share one identical probability — the "top 50" is then an arbitrary slice of a large tied block. **Precision@K punishes coarse scores even when the underlying ordering is sound**, which is exactly the kind of thing a single AUC would have hidden. The tree stays in the notebook as the readable explanation of *what* the model keys on; it is not the thing to ship.
- **The headline gap is smaller than the pooled table makes it look.** Pooled across all 9,836 out-of-fold pages the model beats the rule by ~6 points of P@50 (+3 pages per reviewer-week). Counted fold by fold, both sit near 0.62–0.64 with a spread of ~0.16, so the model's edge is **inside the fold-to-fold noise**. The directional read: the model is at least as good as the rule and its advantage shows up when it can rank across the whole roster, not when it has to fill 50 slots from one or two small clients. On this data I would call it *not worse, probably better, not yet proven better*.
- **The queue got more concentrated, not less.** The largest client holds ~40% of the universe and takes ~72% of the model's top 50 against ~68% of the rule's; the model spreads its 50 across 5 clients where the rule reaches 7. The Week-4 warning stands and the model slightly worsens it — a reviewer should still ask whether a cause is site-wide before rewriting page by page.
- **The false positives are self-repairing pages**, not misread ones. Only 3 of the model's top 50 were wrong, and all three were pages whose CTR climbed back toward the band norm on its own between windows — two of them alongside a large drop in impressions. That is the honest cost of one window of history: the data cannot tell "durably broken" from "temporarily dipping", and a longer panel (the warehouse, Week 6) is the fix, not a bigger model.
- **Probabilities are usable but optimistic-to-flat at the top.** The decile table tracks the observed rate closely up to the ninth decile, then predicts ~0.55 where ~0.49 actually persists. Fine for ordering a queue; not something to quote as "a 55% chance this page is broken".

**Limits I am not writing around:** 14.6% of eligible pages fell below the volume floor in the later window and were dropped, so every number here is conditional on a page still being visible a month later — collapses are out of scope by construction. The label is a defined-rule proxy for persistence, not a reviewed outcome. And 28 clients over two 30-day windows is a small, unbalanced panel.

**What none of this says:** that these pages will improve, that a title rewrite causes clicks, or that the pattern extends past these clients and these windows. The measured claim is narrower: *ranked by this model, a reviewer's 50 slots land on pages whose click shortfall was still there a month later at least as often as the Week-4 rule's 50 slots did, and pooled across the full roster, somewhat more often.*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [24]:
checks = {
    "baseline and model are in one table, same rows, same metric, same run":
        {"Rule baseline (Week 4)", "Base rate (random order)"} <= set(table.index),
    "split is grouped by client -- no client on both sides of any fold":
        all(set(lane.client_id.iloc[tr]).isdisjoint(set(lane.client_id.iloc[te])) for tr, te in folds),
    "label comes from a later window than every feature":
        not offenders,
    "base rate reported next to precision@K":
        "Base rate (random order)" in table.index,
    "model vs baseline reported at the primary metric (P@50), pooled AND per fold":
        {"P@50", "P@50 per-fold sd"} <= set(table.columns) and len(fold_p50["Rule baseline (Week 4)"]) == 5,
    "error analysis names concrete wrong cases": len(wrong) == 3,
    "simplest model reported next to the most complex one":
        {"Decision Tree (d=3)", "Gradient Boosting"} <= set(table.index),
    "seeds fixed and library versions recorded": SEED == 42,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values())

metrics = {
    "notebook": "w05_model.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "design": {"features_window": "prev_30d + static metadata", "label_window": "last_30d",
               "label": "persisted_shortfall: ctr_last < 0.5 x band norm (later window), expected clicks >= 10/3",
               "split": "GroupKFold(5) on client_id, out-of-fold predictions",
               "declared_contamination": "avg_position is a 90d average overlapping the outcome window; no-position refit reported",
               "seed": SEED, "sklearn": sklearn.__version__},
    "universe": {"pages": len(lane), "clients": int(lane.client_id.nunique()),
                 "dropped_not_eligible_later_window": dropped, "base_rate": round(float(base_rate), 4)},
    "results": json.loads(table.round(4).to_json(orient="index")),
    "best_model": best,
    "p50_per_fold": {k: [round(v, 4) for v in vs] for k, vs in fold_p50.items()},
    "top_features_permutation": imp.head(5)[["feature", "drop_in_PR_AUC"]].round(4).to_dict("records"),
    "no_position_refit": json.loads(sens.round(4).to_json(orient="index")),
}
(OUT / "w05_model_metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"\nWrote {OUT / 'w05_model_metrics.json'}")

PASS - baseline and model are in one table, same rows, same metric, same run
PASS - split is grouped by client -- no client on both sides of any fold
PASS - label comes from a later window than every feature
PASS - base rate reported next to precision@K
PASS - model vs baseline reported at the primary metric (P@50), pooled AND per fold
PASS - error analysis names concrete wrong cases
PASS - simplest model reported next to the most complex one
PASS - seeds fixed and library versions recorded

Wrote C:\Users\muham\OneDrive\Desktop\Flyrankkk\work\outputs\w05_model_metrics.json
